# RoboVision Restore: simulated robot views
Run `01_start_here.ipynb` first. This optional notebook collects first-person
RGB views and true instance boxes in AI2-THOR, then runs the same imaging experiment.

Select **Runtime -> Change runtime type -> GPU**. AI2-THOR requires graphics
support in addition to PyTorch CUDA support. Its simulator download is much
larger than the main notebook's six images. GPU availability is not guaranteed.
The hosted rendering path is not yet verified by the recorded local CPU run.

This is a perception dataset collector, not a navigation policy.


In [ ]:
import os, sys, subprocess, importlib.util

# Colab already has a compatible torch/torchvision pair. Preserve it.
# ROBOVISION_SKIP_INSTALL is only used by the local notebook verification script.
if os.environ.get("ROBOVISION_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "numpy>=1.26,<3", "Pillow>=10,<13", "scipy>=1.12,<2",
                           "scikit-image>=0.24,<1", "matplotlib>=3.8,<4"])
    if importlib.util.find_spec("torch") is None or importlib.util.find_spec("torchvision") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision"])
try:
    import torch, torchvision
except (ImportError, RuntimeError) as error:
    raise RuntimeError("Torch/torchvision could not load. In Colab, disconnect and delete the runtime, then rerun from the first cell.") from error
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__, "Torchvision:", torchvision.__version__)
print("GPU available:", torch.cuda.is_available())


In [ ]:
from pathlib import Path
import base64, hashlib, io, os, sys, zipfile

# This is a generated source snapshot, not downloaded executable code.
# The readable source lives in the project repository and is unpacked below.
PAYLOAD = "..."
EXPECTED_SHA256 = "7cd6842b18119c51acd6a7f4f954f2c9a4cc556c7bdfd2684a5321c0b79af1bb"
archive_bytes = base64.b64decode(PAYLOAD)
assert hashlib.sha256(archive_bytes).hexdigest() == EXPECTED_SHA256
PROJECT = Path("/content/robovision_restore") if Path("/content").exists() else Path.cwd() / "robovision_colab_workspace"
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    for member in archive.infolist():
        destination = (PROJECT / member.filename).resolve()
        if not destination.is_relative_to(PROJECT.resolve()):
            raise ValueError("Archive path escapes the project directory")
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(member.filename))
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT.resolve()))
print("Project ready:", PROJECT)


## 1. Prepare simulator rendering
The official headless API uses `CloudRendering`. If the assigned Colab runtime
cannot provide Vulkan graphics, use the official
[AI2-THOR Colab setup](https://github.com/allenai/ai2thor-colab) to start an X server
and set `CLOUD_RENDERING = False` below. The main imaging notebook remains usable.


In [ ]:
import platform, shutil, subprocess, sys
if platform.system() != "Linux" or shutil.which("nvidia-smi") is None:
    raise RuntimeError("This optional notebook requires a Linux GPU runtime. In Colab select a GPU runtime.")
subprocess.run(["nvidia-smi"], check=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ai2thor==5.0.0"])
subprocess.check_call(["apt-get", "update", "-qq"])
subprocess.check_call(["apt-get", "install", "-y", "-qq", "libvulkan1", "vulkan-tools"])


## 2. Collect labeled views with a scene-level split
Development scenes: FloorPlan1 and FloorPlan5.
Test scenes: FloorPlan2 and FloorPlan6.
This prevents adjacent views from the same scene appearing in both splits.

Labels come from visible instance masks. Mug/Cup are mapped to COCO cup.
These visible-mask boxes and category conventions differ from COCO annotations.
The collector samples camera poses and retains views with a target object.


In [ ]:
from robovision.ai2thor_data import collect
CLOUD_RENDERING = True
manifest_path = collect(per_scene=6, cloud_rendering=CLOUD_RENDERING)
print("Saved robot-view manifest:", manifest_path)


## 3. Evaluate the same restoration methods
Use all collected views so both development and test scenes are represented.
The COCO-pretrained detector may perform poorly in simulation even on clean
images. Report that clean reference and the domain gap explicitly.


In [ ]:
from pathlib import Path
from IPython.display import display, Markdown, Image as DisplayImage
from robovision.experiment import Config, run
result = run(Config(manifest=str(manifest_path), output="runs/robot_views", quick=False))
output = Path(result["output"])
display(Markdown((output / "RESULTS.md").read_text()))
display(DisplayImage(filename=str(output / "metrics.png")))
display(DisplayImage(filename=str(output / "detection_comparison.png")))


## 4. Export the views and evidence
This saves the images, labels, run configuration, plots, and notes.
To support broader conclusions, collect more independent scenes and repeat
noise seeds. The present scene split is still a small pilot.


In [ ]:
import zipfile
archive_path = Path("robovision_robot_views.zip")
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for folder in (Path("data/ai2thor"), output, Path("docs")):
        for path in folder.rglob("*"):
            if path.is_file(): archive.write(path, path.as_posix())
    archive.write("THIRD_PARTY.md")
from google.colab import files
files.download(str(archive_path))
